In [42]:
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import \
    metrics_search_for_two_fragments_df
from time import sleep
import pvlib
import pytz
from datetime import date, datetime, timedelta

In [43]:
pd.__version__

'2.3.3'

In [44]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records', 'sample_year'],
      dtype='object')

In [4]:
systems_cleaned.loc[5:10]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
5,34,Andre Agassi Preparatory Academy - Building A,"Las Vegas, NV",America/Los_Angeles,36.1952,-115.1582,620.0,146.64,Bwh,26,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,3626,2011
6,35,Andre Agassi Preparatory Academy - Gymnasium,"Las Vegas, NV",America/Los_Angeles,36.1952,-115.1582,620.0,121.68,Bwh,26,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,3625,2011
7,36,NREL low-X x-Si -1,"Golden, CO",7,39.7040,-105.1773,1794.7,2.21,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,2516,2013
8,50,NREL x-Si 6,"Golden, CO",7,39.7420,-105.1727,1994.7,6.00,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,8455,1995
9,51,NREL x-Si 7,"Golden, CO",7,39.7416,-105.1734,1994.7,6.00,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,8032,1995
10,1199,Distributed Sun - Hunt Valley,"Cockeysville, MD",America/New_York,39.4856,-76.6636,325.0,52.92,Cfa,34,...,True,True,True,True,True,multi-Si,multicrystalline_Si,PVDAQ General,3706,2011


In [45]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [46]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

In [7]:
systems_cleaned_enough_ac_data = systems_cleaned[systems_cleaned['system_id'].isin(enough_data_parquet_power_systems)]

In [48]:
systems_cleaned_enough_ac_data.head()

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
2,4,NREL x-Si -1,"Golden, CO",7,39.7406,-105.1774,1795.3,1.00,BSk,12,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,5063,2008
3,10,NREL CIS -1,"Golden, CO",7,39.7404,-105.1774,1792.8,1.12,BSk,12,...,True,True,True,True,True,cis family thin-film,thin_film,PVDAQ General,5893,2007
4,33,Silicor Materials,"Golden, CO",7,39.7404,-105.1772,1794.0,2.40,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,4438,2011
5,34,Andre Agassi Preparatory Academy - Building A,"Las Vegas, NV",America/Los_Angeles,36.1952,-115.1582,620.0,146.64,Bwh,26,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,3626,2011
6,35,Andre Agassi Preparatory Academy - Gymnasium,"Las Vegas, NV",America/Los_Angeles,36.1952,-115.1582,620.0,121.68,Bwh,26,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,3625,2011


In [9]:
enough_data_locs = systems_cleaned_enough_ac_data[['latitude', 'longitude']].drop_duplicates()

In [10]:
practice_loc = (systems_cleaned_enough_ac_data.at[2, 'latitude'], systems_cleaned_enough_ac_data.at[2, 'longitude'])

In [11]:
nsrdb_api = input('What is your NSRDB API key?')

In [12]:
my_email = input('What is your e-mail address?')

In [15]:
my_system_4_year_2007_weather, my_syster_4_year_2007_weather_meta = pvlib.iotools.get_nsrdb_psm4_aggregated(
    latitude=practice_loc[0],
    longitude=practice_loc[1],
    api_key=nsrdb_api,
    email = my_email,
    year=2007,
    time_step=60,
    parameters=['clearsky_dhi', 'clearsky_dni', 'clearsky_ghi', 'dhi', 'dni', 'ghi',
                'solar_zenith_angle'],
    leap_day=True
)

In [54]:
my_syster_4_year_2007_weather_meta

{'Source': 'NSRDB',
 'Location ID': '479494',
 'City': '-',
 'State': '-',
 'Country': '-',
 'Time Zone': -7,
 'Local Time Zone': -7,
 'Clearsky DHI Units': 'w/m2',
 'Clearsky DNI Units': 'w/m2',
 'Clearsky GHI Units': 'w/m2',
 'Dew Point Units': 'c',
 'DHI Units': 'w/m2',
 'DNI Units': 'w/m2',
 'GHI Units': 'w/m2',
 'Solar Zenith Angle Units': 'Degree',
 'Temperature Units': 'c',
 'Pressure Units': 'mbar',
 'Relative Humidity Units': '%',
 'Precipitable Water Units': 'cm',
 'Wind Direction Units': 'Degrees',
 'Wind Speed Units': 'm/s',
 'Cloud Type -15': 'N/A',
 'Cloud Type 0': 'Clear',
 'Cloud Type 1': 'Probably Clear',
 'Cloud Type 2': 'Fog',
 'Cloud Type 3': 'Water',
 'Cloud Type 4': 'Super-Cooled Water',
 'Cloud Type 5': 'Mixed',
 'Cloud Type 6': 'Opaque Ice',
 'Cloud Type 7': 'Cirrus',
 'Cloud Type 8': 'Overlapping',
 'Cloud Type 9': 'Overshooting',
 'Cloud Type 10': 'Unknown',
 'Cloud Type 11': 'Dust',
 'Cloud Type 12': 'Smoke',
 'Fill Flag 0': 'N/A',
 'Fill Flag 1': 'Missing Im

In [ ]:
my_system_4_year_2007_weather.head()

,Year,Month,Day,Hour,Minute,dhi_clear,dni_clear,ghi_clear,dhi,dni,ghi,solar_zenith
2007-01-01 00:30:00-07:00,2007,1,1,0,30,0.0,0.0,0.0,0.0,0.0,0.0,162.42
2007-01-01 01:30:00-07:00,2007,1,1,1,30,0.0,0.0,0.0,0.0,0.0,0.0,155.32
2007-01-01 02:30:00-07:00,2007,1,1,2,30,0.0,0.0,0.0,0.0,0.0,0.0,145.00
2007-01-01 03:30:00-07:00,2007,1,1,3,30,0.0,0.0,0.0,0.0,0.0,0.0,133.70
2007-01-01 04:30:00-07:00,2007,1,1,4,30,0.0,0.0,0.0,0.0,0.0,0.0,122.18


In [19]:
my_system_4_year_2007_weather[my_system_4_year_2007_weather['Month']==5]

,Year,Month,Day,Hour,Minute,dhi_clear,dni_clear,ghi_clear,dhi,dni,ghi,solar_zenith
2007-05-01 00:30:00-07:00,2007,5,1,0,30,0.0,0.0,0.0,0.0,0.0,0.0,124.77
2007-05-01 01:30:00-07:00,2007,5,1,1,30,0.0,0.0,0.0,0.0,0.0,0.0,121.21
2007-05-01 02:30:00-07:00,2007,5,1,2,30,0.0,0.0,0.0,0.0,0.0,0.0,114.80
2007-05-01 03:30:00-07:00,2007,5,1,3,30,0.0,0.0,0.0,0.0,0.0,0.0,106.32
2007-05-01 04:30:00-07:00,2007,5,1,4,30,0.0,0.0,0.0,0.0,0.0,0.0,96.43
...,...,...,...,...,...,...,...,...,...,...,...,...
2007-05-31 19:30:00-07:00,2007,5,31,19,30,0.0,0.0,0.0,0.0,0.0,0.0,92.19
2007-05-31 20:30:00-07:00,2007,5,31,20,30,0.0,0.0,0.0,0.0,0.0,0.0,101.46
2007-05-31 21:30:00-07:00,2007,5,31,21,30,0.0,0.0,0.0,0.0,0.0,0.0,109.22
2007-05-31 22:30:00-07:00,2007,5,31,22,30,0.0,0.0,0.0,0.0,0.0,0.0,114.91


In [17]:
my_system_4_year_2007_weather.dtypes

Year              int64
Month             int64
Day               int64
Hour              int64
Minute            int64
dhi_clear       float64
dni_clear       float64
ghi_clear       float64
dhi             float64
dni             float64
ghi             float64
solar_zenith    float64
dtype: object

In [20]:
my_system_4_reset_index = my_system_4_year_2007_weather.reset_index()

In [27]:
my_system_4_reset_index['index'].dtype

datetime64[ns, Etc/GMT+7]

In [25]:
my_system_4_reset_index['time_pandas'] = pd.to_datetime(my_system_4_reset_index['index'])

In [26]:
my_system_4_reset_index.head()

,index,Year,Month,Day,Hour,Minute,dhi_clear,dni_clear,ghi_clear,dhi,dni,ghi,solar_zenith,time_pandas
0,2007-01-01 00:30:00-07:00,2007,1,1,0,30,0.0,0.0,0.0,0.0,0.0,0.0,162.42,2007-01-01 00:30:00-07:00
1,2007-01-01 01:30:00-07:00,2007,1,1,1,30,0.0,0.0,0.0,0.0,0.0,0.0,155.32,2007-01-01 01:30:00-07:00
2,2007-01-01 02:30:00-07:00,2007,1,1,2,30,0.0,0.0,0.0,0.0,0.0,0.0,145.00,2007-01-01 02:30:00-07:00
3,2007-01-01 03:30:00-07:00,2007,1,1,3,30,0.0,0.0,0.0,0.0,0.0,0.0,133.70,2007-01-01 03:30:00-07:00
4,2007-01-01 04:30:00-07:00,2007,1,1,4,30,0.0,0.0,0.0,0.0,0.0,0.0,122.18,2007-01-01 04:30:00-07:00


In [50]:
my_system_4_reset_index[
    (my_system_4_reset_index['Month'] == 3)
    & (my_system_4_reset_index['Day'] == 11)]

,index,Year,Month,Day,Hour,Minute,dhi_clear,dni_clear,ghi_clear,dhi,dni,ghi,solar_zenith,time_pandas
1656,2007-03-11 00:30:00-07:00,2007,3,11,0,30,0.0,0.0,0.0,0.0,0.0,0.0,143.84,2007-03-11 00:30:00-07:00
1657,2007-03-11 01:30:00-07:00,2007,3,11,1,30,0.0,0.0,0.0,0.0,0.0,0.0,139.87,2007-03-11 01:30:00-07:00
1658,2007-03-11 02:30:00-07:00,2007,3,11,2,30,0.0,0.0,0.0,0.0,0.0,0.0,132.26,2007-03-11 02:30:00-07:00
1659,2007-03-11 03:30:00-07:00,2007,3,11,3,30,0.0,0.0,0.0,0.0,0.0,0.0,122.52,2007-03-11 03:30:00-07:00
1660,2007-03-11 04:30:00-07:00,2007,3,11,4,30,0.0,0.0,0.0,0.0,0.0,0.0,111.64,2007-03-11 04:30:00-07:00
1661,2007-03-11 05:30:00-07:00,2007,3,11,5,30,0.0,0.0,0.0,0.0,0.0,0.0,100.24,2007-03-11 05:30:00-07:00
1662,2007-03-11 06:30:00-07:00,2007,3,11,6,30,13.0,253.0,20.0,13.0,253.0,20.0,88.43,2007-03-11 06:30:00-07:00
1663,2007-03-11 07:30:00-07:00,2007,3,11,7,30,41.0,745.0,204.0,41.0,745.0,204.0,77.32,2007-03-11 07:30:00-07:00
1664,2007-03-11 08:30:00-07:00,2007,3,11,8,30,53.0,907.0,414.0,53.0,907.0,414.0,66.59,2007-03-11 08:30:00-07:00
1665,2007-03-11 09:30:00-07:00,2007,3,11,9,30,60.0,993.0,602.0,60.0,993.0,602.0,56.94,2007-03-11 09:30:00-07:00


### OK, let us practice System 34 (named timezone vs. UTC timezone)

In [10]:
system_id = 34
my_year = 2011
my_basic_dir = '../../../../data_ds_project/testing_yearly_parquet/'
my_direct_dir = Path(my_basic_dir + f'{system_id}/')
my_year_pq = pq.ParquetDataset(
    my_direct_dir,
    filters=[('year', '==', my_year)],
)
my_year_df = my_year_pq.read().to_pandas()


In [11]:
system_34_rows = systems_cleaned_enough_ac_data[systems_cleaned_enough_ac_data['system_id'] == 34]
system_34_rows

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
5,34,Andre Agassi Preparatory Academy - Building A,"Las Vegas, NV",America/Los_Angeles,36.1952,-115.1582,620.0,146.64,Bwh,26,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,3626,2011


In [12]:
my_year_df['time'].dtype

dtype('<M8[ns]')

Check my system

In [13]:
np.dtype('datetime64[ns]') == np.dtype('<M8[ns]')

True

As per <https://stackoverflow.com/questions/29206612/difference-between-data-type-datetime64ns-and-m8ns>, M8[ns] with a less-than/greater-than symbol are subclasses of datetime64[ns] (almost implementations) with a particular little-endian/big-endian structure.

In [35]:
my_year_df.head()

,time,ac_power_kW,year
0,2011-01-01 00:00:00,-0.2,2011
1,2011-01-01 00:15:00,-0.3,2011
2,2011-01-01 00:30:00,-0.3,2011
3,2011-01-01 00:45:00,-0.2,2011
4,2011-01-01 01:00:00,-0.3,2011


In [14]:
mar_11_data = my_year_df[my_year_df['time'].dt.date==date(2011, 3, 13)]

In [15]:
mar_11_data.iloc[40:60]

,time,ac_power_kW,year
6838,2011-03-13 10:00:00,93.2,2011
6839,2011-03-13 10:15:00,95.8,2011
6840,2011-03-13 10:30:00,98.8,2011
6841,2011-03-13 10:45:00,97.4,2011
6842,2011-03-13 11:00:00,93.1,2011
6843,2011-03-13 11:15:00,87.4,2011
6844,2011-03-13 11:30:00,94.7,2011
6845,2011-03-13 11:45:00,85.7,2011
6846,2011-03-13 12:00:00,101.2,2011
6847,2011-03-13 12:15:00,93.9,2011


In [57]:
nov_6_data = my_year_df[my_year_df['time'].dt.date==date(2011, 11, 6)]

In [60]:
nov_6_data.iloc[4:16]

,time,ac_power_kW,year
29645,2011-11-06 01:00:00,-0.3,2011
29646,2011-11-06 01:15:00,-0.3,2011
29647,2011-11-06 01:30:00,-0.2,2011
29648,2011-11-06 01:45:00,-0.3,2011
29649,2011-11-06 02:00:00,-0.3,2011
29650,2011-11-06 02:15:00,-0.2,2011
29651,2011-11-06 02:30:00,-0.2,2011
29652,2011-11-06 02:45:00,-0.2,2011
29653,2011-11-06 03:00:00,-0.3,2011
29654,2011-11-06 03:15:00,-0.3,2011


In [16]:
my_year_df['time_better'] = my_year_df['time'].dt.tz_localize(system_34_rows.at[5, 'timezone_or_utc_offset'])

NonExistentTimeError: 2011-03-13 02:00:00

In [49]:
my_timezones = set(systems_cleaned_enough_ac_data['timezone_or_utc_offset'].unique())
my_timezones

{'5',
 '7',
 'America/Chicago',
 'America/Denver',
 'America/Los_Angeles',
 'America/New_York'}

In [50]:
timezone_sample_systems = {
    time_zone: (0, 0) for time_zone in my_timezones
}
for time_zone in my_timezones:
    systems_this_timezone = systems_cleaned_enough_ac_data[systems_cleaned_enough_ac_data['timezone_or_utc_offset'] == time_zone]
    first_ind = systems_this_timezone.index[0]
    timezone_sample_systems[time_zone] = (int(first_ind), int(systems_cleaned_enough_ac_data.at[first_ind, 'system_id']))

In [51]:
timezone_sample_systems

{'7': (2, 4),
 'America/Chicago': (35, 1244),
 'America/Denver': (150, 1430),
 'America/Los_Angeles': (5, 34),
 '5': (156, 4901),
 'America/New_York': (10, 1199)}

In [17]:
before_after_dst = {
    'America/Los_Angeles': (8, 7),
    'America/Denver': (7, 6),
    'America/Chicago': (6, 5),
    'America/New_York': (5, 4)
}

In [21]:
def is_good_timezone(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    before_after_dst: dict,
    print_messages: bool = True
):
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_ind = relevant_rows_systems.index[0]
    sample_year = int(systems_cleaned.at[first_ind, 'sample_year'])
    nominal_timezone = systems_cleaned.at[first_ind, 'timezone_or_utc_offset']
    if len(nominal_timezone) == 1:  #single-digit
        nominal_timezone = f'Etc/GMT+{nominal_timezone}'

    # grab data from sample year
    core_dir = Path(input_dir + f'{system_id}/')
    sample_year_pq = pq.ParquetDataset(
        core_dir,
        filters = [('year', '==', sample_year)]
    )
    sample_year_df = sample_year_pq.read().to_pandas()
    try:
        sample_year_df['time'] = sample_year_df['time'].dt.tz_localize(nominal_timezone)
        if print_messages:
            print(f'System {system_id} has a nice timezone of {nominal_timezone}.')
        return True
    except (pytz.NonExistentTimeError, pytz.AmbiguousTimeError):
        options_tuple = before_after_dst[nominal_timezone]
        if print_messages:
            print(f'System {system_id} between the options in GMT+{options_tuple}')
        return False
    except BaseException as e:
        raise e


In [94]:
for key in timezone_sample_systems.keys():
    (ind, system_id) = timezone_sample_systems[key]
    is_good_timezone(
        '../../../../data_ds_project/testing_yearly_parquet/',
        system_id,
        systems_cleaned,
        before_after_dst
    )  

System 1244 between the options in GMT+(6, 5)
System 4 has a nice timezone of Etc/GMT+7.
System 1199 has a nice timezone of America/New_York.
System 4901 has a nice timezone of Etc/GMT+5.
System 1430 between the options in GMT+(7, 6)
System 34 between the options in GMT+(8, 7)


In [95]:
for system_id in enough_data_parquet_power_list:
    is_good_timezone(
        '../../../../data_ds_project/testing_yearly_parquet/',
        system_id,
        systems_cleaned,
        before_after_dst
    )

System 4 has a nice timezone of Etc/GMT+7.
System 10 has a nice timezone of Etc/GMT+7.
System 33 has a nice timezone of Etc/GMT+7.
System 34 between the options in GMT+(8, 7)
System 35 between the options in GMT+(8, 7)
System 36 has a nice timezone of Etc/GMT+7.
System 50 has a nice timezone of Etc/GMT+7.
System 51 has a nice timezone of Etc/GMT+7.
System 1199 has a nice timezone of America/New_York.
System 1200 has a nice timezone of America/New_York.
System 1201 between the options in GMT+(5, 4)
System 1202 between the options in GMT+(5, 4)
System 1203 between the options in GMT+(5, 4)
System 1204 has a nice timezone of America/New_York.
System 1208 has a nice timezone of Etc/GMT+7.
System 1214 between the options in GMT+(5, 4)
System 1217 between the options in GMT+(5, 4)
System 1219 between the options in GMT+(5, 4)
System 1221 between the options in GMT+(5, 4)
System 1223 between the options in GMT+(5, 4)
System 1225 between the options in GMT+(5, 4)
System 1231 between the option

In [29]:
def is_good_timezone_every_year(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    before_after_dst: dict,
    print_messages: bool = True
):
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_ind = relevant_rows_systems.index[0]
    nominal_timezone = systems_cleaned.at[first_ind, 'timezone_or_utc_offset']
    if len(nominal_timezone) == 1:  #single-digit
        nominal_timezone = f'Etc/GMT+{nominal_timezone}'

    
    start_year = 1998
    end_year = 2023
    num_years = end_year - start_year + 1
    # make output range
    data_per_year = pd.DataFrame(
        data = np.full((num_years, 3), pd.NA),
        index = range(start_year, end_year + 1),
        columns= ['good_timezone', 'start_date', 'end_date']
    ) 
    # grab data from sample year
    core_dir = Path(input_dir + f'{system_id}/')  
    for year in range(start_year, end_year + 1):
        current_year_pq = pq.ParquetDataset(
            core_dir,
            filters = [('year', '==', year)]
        )
        current_year_df = current_year_pq.read().to_pandas()
        if current_year_df.shape[0] > 10:
            current_year_df.loc[:, 'date'] = current_year_df['time'].dt.date
            data_per_year.at[year, 'start_date'] = current_year_df['date'].min()
            data_per_year.at[year, 'end_date'] = current_year_df['date'].max()
            # no way of checking if the time too short
            if (
                (data_per_year.at[year, 'end_date'] - data_per_year.at[year, 'start_date'])
                > timedelta(days=30)
            ):
                try:
                    current_year_df['time'] = current_year_df['time'].dt.tz_localize(nominal_timezone)
                    if print_messages:
                        print(f'System {system_id} has a nice timezone of {nominal_timezone} '
                              + f'in the year {year}.')
                    data_per_year.at[year, 'good_timezone'] = True
                except (pytz.NonExistentTimeError, pytz.AmbiguousTimeError):
                    options_tuple = before_after_dst[nominal_timezone]
                    if print_messages:
                        print(f'System {system_id}, year {year}, lies between the options in GMT+{options_tuple}')
                    data_per_year.at[year, 'good_timezone'] = False
                except BaseException as e:
                    raise e
    # drop empty rows
    data_per_year = data_per_year.dropna(axis=0, how='all')
    return data_per_year


In [30]:
is_good_timezone_every_year(
    '../../../../data_ds_project/testing_yearly_parquet/',
    1200,
    systems_cleaned,
    before_after_dst,
    True
)

System 1200 has a nice timezone of America/New_York in the year 2010.
System 1200 has a nice timezone of America/New_York in the year 2011.
System 1200 has a nice timezone of America/New_York in the year 2012.
System 1200 has a nice timezone of America/New_York in the year 2013.
System 1200, year 2014, lies between the options in GMT+(5, 4)
System 1200, year 2015, lies between the options in GMT+(5, 4)
System 1200, year 2016, lies between the options in GMT+(5, 4)
System 1200, year 2017, lies between the options in GMT+(5, 4)
System 1200, year 2018, lies between the options in GMT+(5, 4)
System 1200, year 2019, lies between the options in GMT+(5, 4)
System 1200 has a nice timezone of America/New_York in the year 2020.


,good_timezone,start_date,end_date
2010,True,2010-10-03,2010-12-31
2011,True,2011-01-01,2011-12-31
2012,True,2012-01-01,2012-12-13
2013,True,2013-01-20,2013-12-31
2014,False,2014-01-01,2014-12-31
2015,False,2015-01-01,2015-12-31
2016,False,2016-01-01,2016-12-31
2017,False,2017-01-01,2017-12-31
2018,False,2018-01-01,2018-12-31
2019,False,2019-01-01,2019-12-31


In [38]:
systems_with_some_bad_timestamps = []
for system_id in enough_data_parquet_power_list:
    timezones_per_year = is_good_timezone_every_year(
        '../../../../data_ds_project/testing_yearly_parquet/',
        system_id,
        systems_cleaned,
        before_after_dst,
        False
    )
    bad_timezones_per_year = timezones_per_year[timezones_per_year['good_timezone'] == False]
    if bad_timezones_per_year.shape[0] > 0:
        systems_with_some_bad_timestamps.append(system_id)
        print(f'System {system_id}')
        print('Bad years:')
        print(timezones_per_year)
        print('')

System 34
Bad years:
     good_timezone  start_date    end_date
2010          True  2010-08-23  2010-12-31
2011         False  2011-01-01  2011-12-31
2012         False  2012-01-01  2012-12-31
2013         False  2013-01-01  2013-12-31
2014         False  2014-01-01  2014-12-31
2015         False  2015-01-01  2015-12-31
2016         False  2016-01-01  2016-12-31
2017         False  2017-01-01  2017-12-31
2018         False  2018-01-01  2018-12-31
2019         False  2019-01-01  2019-12-31
2020          True  2020-01-01  2020-07-26

System 35
Bad years:
     good_timezone  start_date    end_date
2010         False  2010-08-23  2010-12-31
2011         False  2011-01-01  2011-12-31
2012         False  2012-01-01  2012-12-31
2013         False  2013-01-01  2013-12-31
2014         False  2014-01-01  2014-12-31
2015         False  2015-01-01  2015-12-31
2016         False  2016-01-01  2016-12-31
2017         False  2017-01-01  2017-12-31
2018         False  2018-01-01  2018-12-31
2019       

Note, however, that rare "True" years could be caused by a system-down issue at the appropriate timestep.  So need more data to conclude that it is really OK.

In [40]:
len(systems_with_some_bad_timestamps)

61

In [54]:
practice_tz = pytz.timezone('America/New_York')
practice_tz._utc_transition_times

[datetime.datetime(1, 1, 1, 0, 0),
 datetime.datetime(1901, 12, 13, 20, 45, 52),
 datetime.datetime(1918, 3, 31, 7, 0),
 datetime.datetime(1918, 10, 27, 6, 0),
 datetime.datetime(1919, 3, 30, 7, 0),
 datetime.datetime(1919, 10, 26, 6, 0),
 datetime.datetime(1920, 3, 28, 7, 0),
 datetime.datetime(1920, 10, 31, 6, 0),
 datetime.datetime(1921, 4, 24, 7, 0),
 datetime.datetime(1921, 9, 25, 6, 0),
 datetime.datetime(1922, 4, 30, 7, 0),
 datetime.datetime(1922, 9, 24, 6, 0),
 datetime.datetime(1923, 4, 29, 7, 0),
 datetime.datetime(1923, 9, 30, 6, 0),
 datetime.datetime(1924, 4, 27, 7, 0),
 datetime.datetime(1924, 9, 28, 6, 0),
 datetime.datetime(1925, 4, 26, 7, 0),
 datetime.datetime(1925, 9, 27, 6, 0),
 datetime.datetime(1926, 4, 25, 7, 0),
 datetime.datetime(1926, 9, 26, 6, 0),
 datetime.datetime(1927, 4, 24, 7, 0),
 datetime.datetime(1927, 9, 25, 6, 0),
 datetime.datetime(1928, 4, 29, 7, 0),
 datetime.datetime(1928, 9, 30, 6, 0),
 datetime.datetime(1929, 4, 28, 7, 0),
 datetime.datetime(

Okay, so *most* systems have some bad timestamps!

In [61]:
def has_dst_change_days_data(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
):
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_ind = relevant_rows_systems.index[0]
    nominal_timezone = systems_cleaned.at[first_ind, 'timezone_or_utc_offset']
    if len(nominal_timezone) == 1:  #single-digit
        # doesn't have the things I want, but all such systems good anyways!
        return None
    timezone_obj = pytz.timezone(nominal_timezone)
    dst_change_times = timezone_obj._utc_transition_times
    
    start_year = 1998
    end_year = 2023
    num_years = end_year - start_year + 1
    # make output range
    data_per_year = pd.DataFrame(
        data = np.full((num_years, 2), pd.NA),
        index = range(start_year, end_year + 1),
        columns= ['has_start_dst_data', 'has_end_dst_data']
    ) 
    # grab data from sample year
    core_dir = Path(input_dir + f'{system_id}/')  
    for year in range(start_year, end_year + 1):
        current_year_pq = pq.ParquetDataset(
            core_dir,
            filters = [('year', '==', year)]
        )
        current_year_df = current_year_pq.read().to_pandas()
        if current_year_df.shape[0] > 10:
            current_year_df.loc[:, 'date'] = current_year_df['time'].dt.date
            year_transition_dates = [
                my_term.date() for my_term in dst_change_times
                if ((my_term >= datetime(year, 1, 1))
                    and (my_term < datetime(year + 1, 1, 1)))
            ]
            spring_transition_date = year_transition_dates[0]
            fall_transition_date = year_transition_dates[1]
            spring_data = current_year_df[
                current_year_df['date'] == spring_transition_date
            ]
            data_per_year.at[year, 'has_start_dst_data'] = (spring_data.shape[0] > 0)
            fall_data = current_year_df[
                current_year_df['date'] == fall_transition_date
            ]
            data_per_year.at[year, 'has_end_dst_data'] = (fall_data.shape[0] > 0)
    # drop empty rows
    data_per_year = data_per_year.dropna(axis=0, how='all')
    return data_per_year


In [62]:
for system_id in enough_data_parquet_power_list:
    print(f'System {system_id}')
    days_checker = has_dst_change_days_data(
        '../../../../data_ds_project/testing_yearly_parquet/',
        system_id,
        systems_cleaned
    )
    if days_checker is not None:
        print(days_checker)
    else:
        print('No-DST-timezone, all good.')
    print('')

System 4
No-DST-timezone, all good.

System 10
No-DST-timezone, all good.

System 33
No-DST-timezone, all good.

System 34
     has_start_dst_data has_end_dst_data
2010              False            False
2011               True             True
2012               True             True
2013               True             True
2014               True             True
2015               True             True
2016               True             True
2017               True             True
2018               True             True
2019               True             True
2020               True            False

System 35
     has_start_dst_data has_end_dst_data
2010              False             True
2011               True             True
2012               True             True
2013               True             True
2014               True             True
2015               True             True
2016               True             True
2017               True             True
2018 